In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS deltalake.silver
COMMENT 'medallion silver layer';

### Creating dimensional customer and products table - APPLY CHANGES INTO (SCD logic handling) and data quality checks

ALTER TABLE deltalake.silver.products
SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

In [0]:
import dlt
from pyspark.sql.functions import expr, regexp_extract, col, current_timestamp

In [0]:
# APPLY CHANGES INTO target_table
# FROM source_table
# KEYS (key_column[, ...])
# [IGNORE NULL UPDATES]
# [APPLY AS DELETE WHEN condition]
# [APPLY AS TRUNCATE WHEN condition]
# SEQUENCE BY orderByColumn
# [COLUMNS {columnList | * EXCEPT (exceptColumnList)}]
# [STORED AS {SCD TYPE 1 | SCD TYPE 2}]
# [TRACK HISTORY ON {columnList | * EXCEPT (exceptColumnList)}]

# <!-- 
# APPLY CHANGES INTO target_table
# The destination streaming table that will receive the changes.

# FROM source_table
# The source table or stream containing the change data.

# KEYS (key_column[, ...])
# One or more columns that uniquely identify each record (primary key).

# IGNORE NULL UPDATES
# Optional. Ignores updates where all values are null.

# APPLY AS DELETE WHEN condition
# Optional. Specifies a condition to treat certain records as deletes.

# APPLY AS TRUNCATE WHEN condition
# Optional. Specifies a condition to treat certain records as table truncations.

# SEQUENCE BY orderByColumn
# Specifies the column used to order and sequence change events (usually a timestamp or version).

# COLUMNS {columnList | * EXCEPT (exceptColumnList)}
# Optional. Specifies which columns to include or exclude from the change application.

# STORED AS {SCD TYPE 1 | SCD TYPE 2}
# Optional. Specifies whether to use SCD Type 1 (overwrite) or SCD Type 2 (history tracking). Default is SCD Type 1.

# TRACK HISTORY ON {columnList | * EXCEPT (exceptColumnList)}
# Optional. For SCD Type 2, specifies which columns to track for historical changes. -->

In [0]:
#setting expectations for all the silver tables 

__customers_constraints = {
    "valid_customer": "customer_id IS NOT NULL",
    "valid_customer_name": "trim(first_name) <> '' or first_name IS NOT NULL"
}



In [0]:
# read customers, products and owners data as stream tables - CDC is enabled automatically 

# customers silver table

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    },
    comment="Customer Silver Table",
    name="silver_customers"
)
@dlt.expect_all_or_fail(__customers_constraints)
def customers():
    df=(
        spark.readStream.table("deltalake.bronze.customers")
    )

    df_new = df.withColumns({
        "full_name": expr("concat(first_name,' ',last_name)"),
        "email_domain": regexp_extract(col("email"),'@([^\\.]+)',1),
        "last_updated_timestamp": current_timestamp()
    })
    
    return df_new

In [0]:
# read from silver_customers post transformation and create SCD 1 logic for customers table.

dlt.create_streaming_table(
    name="dim_customers",
    comment="Customer Dimension Silver Table",
    table_properties={
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    }
)

dlt.apply_changes(
        target="dim_customers",
        source="LIVE.silver_customers",
        keys=["customer_id"],
        sequence_by = col("registration_date"),
        stored_as_scd_type=1
    )

# updating all the columns if needed specify columnlist or except columns

In [0]:
# Products silver table

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false",
        "delta.feature.timestampNtz": "supported"
    },
    comment="Products Silver Table",
    name="silver_products"
)
def customers():
    df=(
        spark.readStream.table("deltalake.bronze.products")
    )

    df_new = df.withColumn("last_updated_timestamp", current_timestamp())

    return df_new

In [0]:
# read from silver_products and create SCD 2 logic for products table maintaining history

dlt.create_streaming_table(
    name="dim_products",
    comment="Customer Dimension Silver Table",
    table_properties={
        "quality": "silver",
        "pipelines.reset.allowed": "false",
        "delta.feature.timestampNtz": "supported"
    }
)

dlt.apply_changes(
        target="dim_products",
        source="LIVE.silver_products",
        keys=["product_id"],
        sequence_by = col("launch_date"),
        apply_as_deletes=expr("is_active = 'false'"),
        stored_as_scd_type=2
    )

# updating all the columns if needed specify columnlist or except columns

In [0]:
# Orders silver table

@dlt.table(
    table_properties = {
        "quality": "silver",
        "pipelines.reset.allowed": "false"
    },
    comment="Orders Silver Table",
    name="silver_orders"
)
def customers():
    df=(
        spark.readStream.table("deltalake.bronze.orders")
    )
    
    df_new = df.withColumn("last_updated_timestamp", current_timestamp())

    return df_new

In [0]:
# Region Silver table 

# read as a complete Load -> Materialised View

@dlt.table(
    table_properties = {
        "quality": "silver",
    },
    comment="Region Silver Table",
    name="silver_region"
)
def customers():
    df=(
        spark.read.table("deltalake.bronze.region")
    )
    return df

In [0]:
#joined silver table 

@dlt.table(
    name = "fact_policy_transaction",
    table_properties = {
        "quality": "silver",
        "delta.feature.timestampNtz": "supported"
    },
    comment="Silver Final Transaction Table",
)
def fact_policy_transaction():

    df_customers = spark.readStream.table("LIVE.dim_customers")
    df_products = spark.readStream.table("LIVE.dim_products")
    df_orders = spark.readStream.table("LIVE.silver_orders")

    df_joined =(
        df_orders
        .join(df_customers, df_orders.customer_id == df_customers.customer_id, "inner")
        .join(df_products, df_orders.product_id == df_products.product_id, "inner")
        .select(
            df_customers.customer_id.alias("customer_id"),
            df_customers.full_name,
            df_customers.email,
            df_customers.date_of_birth,
            df_customers.email_domain,
            df_customers.address,
            df_customers.phone_number,
            df_customers.registration_date,
            df_products.product_id.alias("product_id"),
            df_products.product_name,
            df_products.product_type,
            df_products.coverage_details,
            df_products.premium_base_rate,
            df_products.launch_date,
            df_products.is_active,
            df_products.__START_AT.alias("start_at"),
            df_products.__END_AT.alias("end_at"),
            df_orders.policy_id.alias("policy_id"),
            df_orders.policy_start_date,
            df_orders.policy_end_date,
            df_orders.premium_amount,
            df_orders.payment_frequency,
            df_orders.policy_status,
            df_orders.last_payment_date,
            df_orders.renewal_date
            )
    )

    return df_joined